In [ ]:
# ================= IMPORTS =================
import os
import glob
import re
import pandas as pd
import numpy as np
import spacy
from indicnlp.tokenize import sentence_tokenize
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ================= CONFIGURATION =================
THRESHOLD = 0.70
WINDOW_SIZE = 30
MIN_SENT_LEN = 10

TEST_DOCS = None     # set to int for testing
MAX_SENTS = None     # set to int for testing

BASE_DIR = r"c:\Users\sunil\Desktop\TELUGU_PROJECT\EMBEDDING\input"
OUTPUT_FILE = "aligned_telugu_eng__output.csv"

# ================= LOAD MODELS =================
print("Loading spaCy (English)...")
nlp = spacy.load("en_core_web_sm", disable=["ner", "parser", "tagger", "lemmatizer"])
nlp.add_pipe("sentencizer")

print("Loading LaBSE model...")
model = SentenceTransformer("sentence-transformers/LaBSE")
print("Models loaded.\n")

# ================= CLEANING FUNCTIONS =================
def clean_telugu_safe(text):
    if not isinstance(text, str):
        return ""
    text = text.replace('\u200b', '').replace('\xa0', ' ')
    text = re.sub(r'పుట\s*\d+|Page\s*\d+', ' ', text)
    text = re.sub(r'[^\u0C00-\u0C7F0-9\s\.,;:\-\(\)\[\]\'"/]+', ' ', text)
    text = re.sub(r'([.,;:\-\(\)\[\]\'"/])\1+', r'\1', text)
    text = re.sub(r'^[\s\.\(\)\[\]\d]+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def clean_english(text):
    if not isinstance(text, str):
        return ""
    text = text.replace('\u200b', '').replace('\xa0', ' ')
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'Page\s*\d+', ' ', text)
    text = re.sub(r'[^A-Za-z0-9\s\.,;:\-\(\)\[\]\'"/]+', ' ', text)
    text = re.sub(r'([.,;:\-\(\)\[\]\'"/])\1+', r'\1', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# ================= COLLECT FILES (ROBUST) =================
all_txt_files = glob.glob(os.path.join(BASE_DIR, "*.txt"))

eng_map = {}
tel_map = {}

for f in all_txt_files:
    name = os.path.basename(f).lower()

    # Extract document ID safely
    m = re.search(r"(\d+)", name)
    if not m:
        continue
    doc_id = m.group(1)

    # Detect English
    if "eng" in name or "english" in name:
        eng_map[doc_id] = f

    # Detect Telugu
    elif "tel" in name or "telugu" in name:
        tel_map[doc_id] = f

print(f"English files found: {len(eng_map)}")
print(f"Telugu files found : {len(tel_map)}")

common_ids = sorted(set(eng_map) & set(tel_map), key=int)

print(f"Found {len(common_ids)} matched document pairs.\n")

if not common_ids:
    raise RuntimeError("❌ No matching ENG/TEL file pairs found. Check filenames!")

# ================= MAIN LOOP =================
all_results = []

doc_ids_to_process = common_ids[:TEST_DOCS] if TEST_DOCS else common_ids

for idx, doc_id in enumerate(doc_ids_to_process, start=1):
    eng_path = eng_map[doc_id]
    tel_path = tel_map[doc_id]

    eng_file = os.path.basename(eng_path)
    tel_file = os.path.basename(tel_path)

    print(f"\nProcessing DocID {doc_id} ({idx}/{len(doc_ids_to_process)})")
    print(f"  ENG: {eng_file}")
    print(f"  TEL: {tel_file}")

    try:
        with open(eng_path, encoding="utf-8") as f:
            raw_eng = f.read()
        with open(tel_path, encoding="utf-8") as f:
            raw_tel = f.read()
    except Exception as e:
        print(f"  ❌ ERROR reading file: {e}")
        continue

    eng_text = clean_english(raw_eng)
    tel_text = clean_telugu_safe(raw_tel)

    eng_sents = [s.text.strip() for s in nlp(eng_text).sents if len(s.text.strip()) > MIN_SENT_LEN]
    tel_sents = [s.strip() for s in sentence_tokenize.sentence_split(tel_text, lang="te")
                 if len(s.strip()) > MIN_SENT_LEN]

    if MAX_SENTS:
        eng_sents = eng_sents[:MAX_SENTS]
        tel_sents = tel_sents[:MAX_SENTS]

    if not eng_sents or not tel_sents:
        print("  ⚠️ Skipped: No valid sentences after cleaning.")
        continue

    print(f"  Eng sentences: {len(eng_sents)}")
    print(f"  Tel sentences: {len(tel_sents)}")

    # ================= EMBEDDINGS =================
    eng_emb = np.atleast_2d(model.encode(eng_sents, batch_size=32, show_progress_bar=False))
    tel_emb = np.atleast_2d(model.encode(tel_sents, batch_size=32, show_progress_bar=False))

    # ================= SIMILARITY & ALIGNMENT =================
    sim_matrix = cosine_similarity(eng_emb, tel_emb)
    doc_ratio = len(tel_sents) / max(1, len(eng_sents))

    last_match_idx = 0
    file_matches = []

    for i in range(len(eng_sents)):
        expected_center = int(i * doc_ratio)
        start_idx = max(last_match_idx, expected_center - WINDOW_SIZE)
        end_idx   = min(len(tel_sents), expected_center + WINDOW_SIZE + 1)

        window_scores = sim_matrix[i, start_idx:end_idx]
        if window_scores.size == 0:
            continue

        local_best_idx = int(np.argmax(window_scores))
        best_score = float(window_scores[local_best_idx])
        global_tel_idx = start_idx + local_best_idx

        if best_score >= THRESHOLD:
            file_matches.append({
                "DocID": doc_id,
                "Filename_English": eng_file,
                "Filename_Telugu": tel_file,
                "English_Reference": eng_sents[i],
                "Telugu": tel_sents[global_tel_idx],
                "Score": round(best_score, 4),
                "Eng_Index": i,
                "Tel_Index": global_tel_idx
            })
            last_match_idx = global_tel_idx

    if file_matches:
        df_temp = pd.DataFrame(file_matches)
        df_temp = df_temp.sort_values("Score", ascending=False)
        df_temp = df_temp.drop_duplicates(subset=["Tel_Index"])
        df_temp = df_temp.drop_duplicates(subset=["Eng_Index"])
        df_temp = df_temp.sort_values("Eng_Index")

        all_results.extend(df_temp.to_dict('records'))
        print(f"  ✅ Final aligned pairs: {len(df_temp)}")
    else:
        print("  ⚠️ No aligned pairs for this document.")

# ================= SAVE =================
df_final = pd.DataFrame(all_results)

if not df_final.empty:
    df_final.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")
    print("\n🎉 ALIGNMENT COMPLETE")
    print(f"Total aligned pairs: {len(df_final)}")
    print(f"Output file: {OUTPUT_FILE}")
else:
    print("\n❌ No alignments found. Check data quality and filenames.")